# 01 データ取得と検証

無料ソース(Stooq 主・yfinance フォールバック・CoinGecko)から少量を取得し、**キャッシュ → データ品質診断 → 検証**の流れを確認する。ネットが無い環境でもセルが落ちないよう、取得失敗時は明示メッセージにフォールバックする(欠損は捏造しない)。

In [1]:
from irp.data import get_prices
from irp.utils.config import load_config

ds = load_config('data_sources')
print('primary:', ds['market']['primary'], '| fallback:', ds['market']['fallback'])

primary: stooq | fallback: yfinance


## 米 ETF を取得して品質診断

In [2]:
import pandas as pd
rows = []
for sym in ['SPY', 'TLT', 'GLD']:
    try:
        r = get_prices(sym, '2018-01-01', '2023-12-31')
        rows.append((sym, r.source, r.quality.rows, r.quality.summary()))
    except Exception as e:
        rows.append((sym, 'OFFLINE', 0, f'download unavailable: {type(e).__name__}'))
pd.DataFrame(rows, columns=['symbol', 'source', 'rows', 'quality']).set_index('symbol')

13:26:16 WARNING irp.data.stooq: download SPY attempt 1 failed: 404 Client Error: Not Found for url: https://stooq.com/q/d/l/?s=spy.us&d1=20180101&d2=20231231&i=d


13:26:17 WARNING irp.data.stooq: download SPY attempt 2 failed: 404 Client Error: Not Found for url: https://stooq.com/q/d/l/?s=spy.us&d1=20180101&d2=20231231&i=d


13:26:18 WARNING irp.data.stooq: download SPY attempt 3 failed: 404 Client Error: Not Found for url: https://stooq.com/q/d/l/?s=spy.us&d1=20180101&d2=20231231&i=d


13:26:18 WARNING irp.data.yfinance: using yfinance FALLBACK for SPY (prefer Stooq; data may differ)


13:26:19 WARNING irp.data.yfinance: SPY: 55 missing business days inside the range (NOT filled)


...

13:26:23 WARNING irp.data.stooq: download GLD attempt 2 failed: 404 Client Error: Not Found for url: https://stooq.com/q/d/l/?s=gld.us&d1=20180101&d2=20231231&i=d


13:26:24 WARNING irp.data.stooq: download GLD attempt 3 failed: 404 Client Error: Not Found for url: https://stooq.com/q/d/l/?s=gld.us&d1=20180101&d2=20231231&i=d


13:26:24 WARNING irp.data.yfinance: using yfinance FALLBACK for GLD (prefer Stooq; data may differ)


13:26:24 WARNING irp.data.yfinance: GLD: 55 missing business days inside the range (NOT filled)


,source,rows,quality
symbol,,,
SPY,yfinance,1509,rows=1509 range=2018-01-02..2023-12-29 max_mis...
TLT,yfinance,1509,rows=1509 range=2018-01-02..2023-12-29 max_mis...
GLD,yfinance,1509,rows=1509 range=2018-01-02..2023-12-29 max_mis...


## crypto(CoinGecko、価格のみ)

In [3]:
from irp.data import get_prices
try:
    btc = get_prices('btc', '2022-01-01', '2023-12-31', source='coingecko')
    print(btc.quality.summary())
    display(btc.data.tail(3))
except Exception as e:
    print('crypto download unavailable (offline?):', type(e).__name__)

13:26:24 WARNING irp.data.coingecko: download btc attempt 1 failed: 401 Client Error: Unauthorized for url: https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range?vs_currency=usd&from=1640995200&to=1703980800


13:26:27 WARNING irp.data.coingecko: download btc attempt 2 failed: 401 Client Error: Unauthorized for url: https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range?vs_currency=usd&from=1640995200&to=1703980800


13:26:29 WARNING irp.data.coingecko: download btc attempt 3 failed: 401 Client Error: Unauthorized for url: https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range?vs_currency=usd&from=1640995200&to=1703980800


crypto download unavailable (offline?): ConnectorError


品質診断は**修復しない**:欠損率・営業日ギャップ・重複・stale を報告するだけ。下流(特徴量/バックテスト)が扱いを決める。